# Notebook 08 — Clasificación de Imagen de Dron con Random Forest
**Teledetección — Maestría en Ingeniería | Universidad del Magdalena**  
**Sesión 4 | Sábado 25 Jul 2026**

Continuamos donde dejamos el Notebook 07. Ahora usamos las **5 bandas del P4M** como variables de entrada para entrenar un clasificador Random Forest que mapea las coberturas del área de la universidad.

### Clases a mapear (universidad del Magdalena)
| Clase | ID | Color |
|-------|-----|-------|
| Agua (lago) | 0 | Azul |
| Suelo / concreto | 1 | Naranja |
| Césped / pastizal | 2 | Verde claro |
| Árboles / arbustos | 3 | Verde oscuro |

### Conexión con tu investigación
Este es exactamente el flujo del paper **CONCAPAN 2022** (Espinosa Valdez, Polo-Castañeda et al.) — solo que ese paper lo aplicó sobre un bananal con el mismo P4M. Las clases cambian (dosel de banano, entre-surco, suelo), pero la metodología RF es idéntica.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import rasterio
import os
import warnings
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, cohen_kappa_score
from sklearn.preprocessing import LabelEncoder
import seaborn as sns
warnings.filterwarnings('ignore')
print('Librerías cargadas')

## Parte 1 — Cargar el ortomosaico del Notebook 07

In [ ]:
RUTA_ORTO = 'ortomosaico_p4m_5bandas.tif'

# Crear imagen de ejemplo si no existe el ortomosaico real
if not os.path.exists(RUTA_ORTO):
    print('Usando imagen de ejemplo (ejecuta primero el Notebook 07 o sube tu ortomosaico)')
    exec(open('crear_ejemplo.py').read()) if os.path.exists('crear_ejemplo.py') else None

    # Crear ejemplo directo aquí
    from rasterio.transform import from_origin
    np.random.seed(42)
    filas, cols = 800, 1200
    lago    = np.zeros((filas,cols),bool); lago[50:250, 50:400]    = True
    arboles = np.zeros((filas,cols),bool); arboles[100:600,700:1150]= True
    cesped  = np.zeros((filas,cols),bool); cesped[300:700,100:700]  = True
    suelo   = ~lago & ~arboles & ~cesped

    refl = {'Blue':[800,7000,2200,1800],'Green':[900,8000,3500,2800],
            'Red': [700,7500,2000,1500],'RE':   [800,7200,5000,9000],'NIR':[600,9000,18000,35000]}
    bandas = []
    for b, vals in refl.items():
        img = np.zeros((filas,cols),np.float32)
        for v, m in zip(vals,[lago,suelo,cesped,arboles]):
            img[m] = v*(1+0.08*np.random.randn(m.sum()))
        bandas.append(img.clip(0).astype(np.uint16))

    t = from_origin(-74.20, 11.00, 1e-6, 1e-6)
    with rasterio.open(RUTA_ORTO,'w',driver='GTiff',height=filas,width=cols,
                       count=5,dtype=np.uint16,crs='EPSG:4326',transform=t) as dst:
        [dst.write(b,i+1) for i,b in enumerate(bandas)]
    print(f'Imagen de ejemplo creada: {RUTA_ORTO}')

with rasterio.open(RUTA_ORTO) as src:
    blue  = src.read(1).astype(np.float32)
    green = src.read(2).astype(np.float32)
    red   = src.read(3).astype(np.float32)
    re    = src.read(4).astype(np.float32)
    nir   = src.read(5).astype(np.float32)
    perfil = src.profile.copy()
    transform_geo = src.transform

filas, cols = blue.shape
print(f'Imagen cargada: {cols} × {filas} px')

## Parte 2 — Definir muestras de entrenamiento

En lugar de dibujar polígonos en un mapa (lo hacemos en GEE), aquí definimos las muestras directamente como coordenadas de píxeles — como si hubiéramos marcado puntos de campo conocidos.

In [ ]:
# Regiones de entrenamiento definidas como recortes (fila_inicio, fila_fin, col_inicio, col_fin)
# Cambia estas coordenadas según tu ortomosaico real
CLASES = {0: 'Agua (lago)', 1: 'Suelo/concreto', 2: 'Césped', 3: 'Árboles'}
COLORES = {0:'#3498DB', 1:'#E67E22', 2:'#A9D18E', 3:'#1B5E20'}

regiones_entrenamiento = [
    # (clase, fila_ini, fila_fin, col_ini, col_fin, nombre)
    (0, 80,  200, 80,  350, 'Lago centro'),
    (0, 150, 230, 200, 390, 'Lago borde'),
    (1, 650, 760, 50,  300, 'Concreto sur'),
    (1, 650, 760, 900, 1100,'Concreto edificio'),
    (1, 700, 790, 400, 700, 'Suelo compactado'),
    (2, 350, 500, 150, 400, 'Césped campo'),
    (2, 400, 600, 400, 650, 'Pastizal central'),
    (3, 150, 450, 750, 1100,'Arbolado derecho'),
    (3, 450, 580, 800, 1100,'Arbustos bajos'),
]

# Construir arrays X (features) y y (etiquetas)
eps = 1e-6
ndvi_arr = (nir - red)  / (nir + red  + eps)
ndre_arr = (nir - re)   / (nir + re   + eps)
ndmi_arr = (nir - blue) / (nir + blue + eps)

# Stack de 8 variables: 5 bandas + 3 índices
stack = np.dstack([blue, green, red, re, nir, ndvi_arr, ndre_arr, ndmi_arr])

X_muestras = []
y_muestras = []

for clase, r0, r1, c0, c1, nombre in regiones_entrenamiento:
    recorte = stack[r0:r1, c0:c1, :].reshape(-1, 8)
    # Muestreo aleatorio de max 500 pixeles por región
    np.random.seed(42)
    idx = np.random.choice(len(recorte), min(500, len(recorte)), replace=False)
    X_muestras.append(recorte[idx])
    y_muestras.append(np.full(len(idx), clase))

X = np.vstack(X_muestras)
y = np.hstack(y_muestras)

print(f'Total muestras: {len(X)} píxeles')
print('Distribución por clase:')
for c_id, c_nom in CLASES.items():
    n = np.sum(y == c_id)
    print(f'  {c_nom}: {n} px ({100*n/len(y):.0f}%)')

In [ ]:
# Visualizar las regiones de entrenamiento sobre la imagen RGB
def normalizar(b, p=2):
    lo, hi = np.percentile(b[b>0], p), np.percentile(b[b>0], 100-p)
    return np.clip((b-lo)/(hi-lo+1e-6), 0, 1)

rgb = np.dstack([normalizar(red), normalizar(green), normalizar(blue)])

fig, ax = plt.subplots(figsize=(12, 7))
ax.imshow(rgb, aspect='auto')

for clase, r0, r1, c0, c1, nombre in regiones_entrenamiento:
    color = COLORES[clase]
    rect = plt.Rectangle((c0, r0), c1-c0, r1-r0, linewidth=2, edgecolor=color,
                          facecolor=color, alpha=0.25)
    ax.add_patch(rect)
    ax.text(c0+3, r0+12, nombre, color='white', fontsize=7,
            bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.7))

leyenda = [mpatches.Patch(color=c, label=n) for n, c in [(v,COLORES[k]) for k,v in CLASES.items()]]
ax.legend(handles=leyenda, loc='lower right', fontsize=9)
ax.set_title('Regiones de entrenamiento sobre ortomosaico P4M', fontsize=12)
ax.axis('off')
plt.tight_layout()
plt.show()

## Parte 3 — Entrenar Random Forest

In [ ]:
VARIABLES = ['Blue','Green','Red','RedEdge','NIR','NDVI','NDRE','NDMI']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

print(f'Entrenamiento: {len(X_train)} muestras')
print(f'Validación:    {len(X_test)} muestras')
print(f'Accuracy train: {rf.score(X_train, y_train)*100:.1f}% (esperado ~100% — RF memoriza)')
print(f'Accuracy test:  {rf.score(X_test,  y_test)*100:.1f}% (este es el que importa)')

In [ ]:
# Reporte completo de clasificación
y_pred = rf.predict(X_test)
kappa = cohen_kappa_score(y_test, y_pred)

print('=== Reporte de clasificación ===')
print(classification_report(y_test, y_pred, target_names=list(CLASES.values())))
print(f'Índice Kappa: {kappa:.4f}')

k_int = 'Casi perfecto' if kappa>0.8 else 'Substancial' if kappa>0.6 else 'Moderado' if kappa>0.4 else 'Regular'
print(f'Interpretación: {k_int}')

In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
nombres_clases = list(CLASES.values())

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=nombres_clases, yticklabels=nombres_clases, ax=ax1, linewidths=0.5)
ax1.set_title('Conteos')
ax1.set_xlabel('Predicho')
ax1.set_ylabel('Real')
plt.setp(ax1.get_xticklabels(), rotation=30, ha='right')

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens', vmin=0, vmax=1,
            xticklabels=nombres_clases, yticklabels=nombres_clases, ax=ax2, linewidths=0.5)
ax2.set_title('Normalizada (Recall por clase)')
ax2.set_xlabel('Predicho')
ax2.set_ylabel('Real')
plt.setp(ax2.get_xticklabels(), rotation=30, ha='right')

fig.suptitle(f'Validación RF — OA: {rf.score(X_test,y_test)*100:.1f}%  Kappa: {kappa:.3f}', fontsize=12)
plt.tight_layout()
plt.show()

## Parte 4 — Importancia de variables

In [ ]:
importancias = rf.feature_importances_
orden = np.argsort(importancias)[::-1]
vars_ord = [VARIABLES[i] for i in orden]
imp_ord  = importancias[orden]

colores_vars = ['#e74c3c' if v in ['NDVI','NDRE','NDMI'] else '#3498db' for v in vars_ord]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(vars_ord, imp_ord, color=colores_vars, alpha=0.85)
for bar, val in zip(bars, imp_ord):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002, f'{val:.3f}',
            ha='center', va='bottom', fontsize=9)

ax.set_ylabel('Importancia relativa')
ax.set_title('Importancia de variables — Random Forest\n(rojo = índices calculados, azul = bandas brutas del P4M)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\nVariable más importante: {vars_ord[0]} ({imp_ord[0]:.3f})')
print(f'Variable menos importante: {vars_ord[-1]} ({imp_ord[-1]:.3f})')

## Parte 5 — Clasificar la imagen completa

In [ ]:
# Aplanar imagen a (N_pixeles × 8_variables)
imagen_plana = stack.reshape(-1, 8)

# Clasificar en lotes para no saturar RAM (importante con imágenes grandes)
print('Clasificando imagen completa...')
LOTE = 500_000
resultado_plano = np.zeros(len(imagen_plana), dtype=np.uint8)

for i in range(0, len(imagen_plana), LOTE):
    lote = imagen_plana[i:i+LOTE]
    resultado_plano[i:i+LOTE] = rf.predict(lote)
    if i % 2_000_000 == 0:
        print(f'  {100*i/len(imagen_plana):.0f}%...')

mapa_clasificado = resultado_plano.reshape(filas, cols)
print('Clasificación completa.')

In [ ]:
# Visualizar el mapa clasificado junto al RGB
from matplotlib.colors import ListedColormap
cmap_custom = ListedColormap([COLORES[i] for i in range(4)])

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(rgb, aspect='auto')
axes[0].set_title('Color Natural (R-G-B)', fontsize=11)
axes[0].axis('off')

im_ndvi = axes[1].imshow(np.clip(ndvi_arr, -0.2, 0.9), cmap='RdYlGn', aspect='auto')
axes[1].set_title('NDVI', fontsize=11)
axes[1].axis('off')
plt.colorbar(im_ndvi, ax=axes[1], fraction=0.046)

im_cls = axes[2].imshow(mapa_clasificado, cmap=cmap_custom, vmin=0, vmax=3, aspect='auto')
axes[2].set_title('Clasificación RF (4 clases)', fontsize=11)
axes[2].axis('off')

parches = [mpatches.Patch(color=COLORES[i], label=CLASES[i]) for i in range(4)]
axes[2].legend(handles=parches, loc='lower right', fontsize=9)

plt.suptitle(f'Ortomosaico P4M — Clasificación RF  |  OA: {rf.score(X_test,y_test)*100:.1f}%  Kappa: {kappa:.3f}',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Estadísticas de área por clase
print('=== Área por clase ===')
total = mapa_clasificado.size
for c_id, c_nom in CLASES.items():
    n = np.sum(mapa_clasificado == c_id)
    # GSD real del P4M a 50 m = 5.29 cm → área por px = 0.0529² m² ≈ 0.0028 m²
    area_m2 = n * (0.0529**2)
    print(f'  {c_nom:<20}: {n:>8,} px = {area_m2:>10,.0f} m² = {area_m2/10000:>6.2f} ha ({100*n/total:.1f}%)')

## Parte 6 — Exportar mapa clasificado como GeoTIFF

In [ ]:
perfil_salida = perfil.copy()
perfil_salida.update({'count': 1, 'dtype': 'uint8', 'nodata': 255})

RUTA_CLASIFICACION = 'clasificacion_rf_dron_universidad.tif'
with rasterio.open(RUTA_CLASIFICACION, 'w', **perfil_salida) as dst:
    dst.write(mapa_clasificado.astype(np.uint8), 1)

print(f'Mapa clasificado exportado: {RUTA_CLASIFICACION}')
print()
print('Para abrir en QGIS:')
print('  1. Layer → Add Raster Layer → selecciona clasificacion_rf_dron_universidad.tif')
print('  2. Layer Properties → Symbology → Paletted/Unique Values')
print('  3. Asigna colores:')
for c_id, c_nom in CLASES.items():
    print(f'     Valor {c_id} → {c_nom} ({COLORES[c_id]})')

---
## Reflexión final

### ¿Qué acabas de hacer?
El flujo completo de **teledetección con drones en Python**:
1. Leer ortomosaico multibanda con `rasterio`
2. Calcular índices espectrales como operaciones NumPy
3. Definir muestras de entrenamiento con conocimiento de campo
4. Entrenar `RandomForestClassifier` con bandas + índices como variables
5. Clasificar toda la imagen en lotes
6. Validar con matriz de confusión y Kappa
7. Exportar el resultado georreferenciado para QGIS

### Conexión con el paper CONCAPAN 2022
En ese paper las clases eran **dosel de banano** y **entre-surco**. El clasificador RF entrenado sobre las 5 bandas del P4M logró separar exactamente esas dos clases con alta precisión, lo que permitió estimar el porcentaje de cobertura del dosel — métrica directa del estado del cultivo.

### Para tu proyecto final
Si usas imágenes de dron, este es el notebook base. Ajusta:
- Las clases (`CLASES`) según lo que quieres mapear
- Las regiones de entrenamiento (`regiones_entrenamiento`) según tu área
- Los hiperparámetros del RF (`n_estimators`, `max_depth`) según el tamaño de tu dataset